# 04 — OCR Ablation

**Question.** Does text recognized from help-center screenshots improve retrieval enough to justify the extra pipeline cost?

The notebook downloads and deduplicates images, runs PaddleOCR, rebuilds only the affected lexical field, and compares with the frozen no-OCR baseline.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
import yaml
if IN_COLAB and not module_available("paddleocr"):
    import torch
    cuda = str(torch.version.cuda or "")
    if torch.cuda.is_available():
        paddle_index = "cu126" if cuda.startswith("12") else "cu118"
        subprocess.check_call([sys.executable,"-m","pip","install","-q","paddlepaddle-gpu==3.2.0",
                               "-i",f"https://www.paddlepaddle.org.cn/packages/stable/{paddle_index}/"])
    else:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","paddlepaddle==3.2.0",
                               "-i","https://www.paddlepaddle.org.cn/packages/stable/cpu/"])
    subprocess.check_call([sys.executable,"-m","pip","install","-q","paddleocr[all]"])

from avito_retriever.data.io import load_articles, load_calibration
from avito_retriever.preprocessing.images import extract_image_manifest, download_images
from avito_retriever.preprocessing.ocr import paddle_ocr_manifest, aggregate_ocr_by_article
from avito_retriever.preprocessing.html import FIELD_COLUMNS
from avito_retriever.preprocessing.normalize import normalize_lexical
from avito_retriever.tokenization.sentencepiece import SentencePieceTokenizer
from avito_retriever.retrieval.bm25f import BM25FIndex
from avito_retriever.evaluation.metrics import evaluate_rankings
from avito_retriever.evaluation.selection import metric_on_split
from avito_retriever.fusion.rrf import weighted_rrf

OUT = result_dir(PROJECT_ROOT, "04_ocr_ablation")
LEX = result_dir(PROJECT_ROOT, "01_sentencepiece_bm25f_search")
HYB = result_dir(PROJECT_ROOT, "02_dense_knn_fusion_search")
articles = load_articles(DATA_DIR); calibration = load_calibration(DATA_DIR)
parsed = pd.read_parquet(PROJECT_ROOT / "artifacts/cache/parsed_articles.parquet")
split = pd.read_csv(LEX / "selection_split.csv")


## Download and cache unique images


In [ ]:
manifest_path = OUT / "download_manifest.parquet"
if manifest_path.exists():
    downloaded = pd.read_parquet(manifest_path)
else:
    manifest = extract_image_manifest(articles)
    downloaded = download_images(manifest, PROJECT_ROOT / "artifacts/images")
    downloaded.to_parquet(manifest_path, index=False)
display(downloaded.download_status.value_counts().rename_axis("status").to_frame("images"))


## Run PaddleOCR (resumable at manifest level)


In [ ]:
ocr_path = OUT / "ocr_manifest.parquet"
if ocr_path.exists():
    ocr_manifest = pd.read_parquet(ocr_path)
else:
    ocr_manifest = paddle_ocr_manifest(downloaded, language="ru", confidence_threshold=0.50)
    ocr_manifest.to_parquet(ocr_path, index=False)
display(ocr_manifest.ocr_status.value_counts().rename_axis("status").to_frame("images"))
print("Recognized non-empty images:", ocr_manifest.ocr_text.fillna("").str.len().gt(0).sum())


## Attach OCR and rerun BM25F


In [ ]:
ocr_by_article = aggregate_ocr_by_article(ocr_manifest)
with_ocr = parsed.drop(columns=["image_ocr"]).merge(ocr_by_article, on="article_id", how="left")
with_ocr["image_ocr"] = with_ocr.image_ocr.fillna("")
with_ocr.to_parquet(OUT / "parsed_articles_with_ocr.parquet", index=False)

best_cfg = yaml.safe_load((LEX / "best_bm25f_config.yaml").read_text())
sp_path = json.loads((LEX / "best_artifacts.json").read_text())["sentencepiece_model"]
sp = SentencePieceTokenizer(sp_path)
normalization = {"unicode_form":"NFKC","lowercase":True,"replace_yo":True,"normalize_quotes":True,"normalize_dashes":True}
lexical = with_ocr.copy()
for field in FIELD_COLUMNS: lexical[field] = lexical[field].fillna("").map(lambda x: normalize_lexical(x, normalization))
queries = calibration.copy(); queries["query_text"] = queries.query_text.map(lambda x: normalize_lexical(x, normalization))
index = BM25FIndex(best_cfg["bm25f"]["fields"], sp.encode, k1=best_cfg["bm25f"]["k1"]).fit(lexical)
ocr_bm25 = index.retrieve(queries, top_k=100, source="bm25f_ocr")
ocr_metrics, ocr_per_query = evaluate_rankings(ocr_bm25, calibration)


## OCR fusion weights


In [ ]:
hybrid = pd.read_parquet(HYB / "best_hybrid_rankings.parquet")
rows, rankings = [], {}
for weight in [0.0,0.25,0.5,0.75,1.0]:
    ranking = hybrid if weight == 0 else weighted_rrf({"hybrid":hybrid,"ocr":ocr_bm25},
                                                       {"hybrid":1.0,"ocr":weight}, rrf_k=40, top_k=100)
    metrics, per_query = evaluate_rankings(ranking, calibration)
    rows.append({"ocr_weight":weight,"tune_map@10":metric_on_split(per_query,split,"ap@10","tune"),**metrics})
    rankings[weight] = (ranking, per_query)
table = pd.DataFrame(rows).sort_values("tune_map@10",ascending=False).reset_index(drop=True)
display(highlight_best(table,["tune_map@10","map@10","recall@50"]))


## Freeze OCR decision


In [ ]:
best = table.iloc[0].to_dict(); ranking, per_query = rankings[best["ocr_weight"]]
ranking.to_parquet(OUT / "best_ocr_rankings.parquet", index=False)
per_query.to_parquet(OUT / "best_ocr_per_query.parquet", index=False)
best["confirm_map@10"] = metric_on_split(per_query,split,"ap@10","confirm")
table.to_csv(OUT / "ocr_leaderboard.csv",index=False); save_json(best,OUT/"best_ocr_config.json")
display(pd.DataFrame([best]))


## Decision rule
Keep OCR only if tune improves and the confirm result does not reverse the gain. Runtime and failed-download rate remain part of the engineering decision.
